In [1]:
#pip install tensorflow

In [2]:
#pip install keras

In [3]:
#pip install scikeras

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from scikeras.wrappers import KerasClassifier

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 1. Data Loading , Expporation and preprocessing

In [ ]:
df = pd.read_csv("sonardataset.csv")
df

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# Encode target label (M/R → 0/1)
label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df.iloc[:, -1])


In [ ]:
# Drop original string label column
df = df.drop(df.columns[-2], axis=1)

In [ ]:
# Split into features and label
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [ ]:
print("Unique encoded classes:", np.unique(y))


In [ ]:
# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Standard scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Preprocessing Completed Successfully!")

In [ ]:
# 2. Base Artificial neural network model

In [ ]:
import warnings 
warnings.filterwarnings('ignore')

In [ ]:
# Build and Train Base Model
base_model = Sequential([
    Dense(32, activation='relu', input_dim=60),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

base_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = base_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    verbose=0
)

In [ ]:
# 3.Hyper parameter tuning

In [ ]:
# Define Hyperparameter Search Model
def create_model(activation='relu', neurons=32, optimizer='adam'):
    model = Sequential()
    model.add(Dense(neurons, activation=activation, input_dim=60))
    model.add(Dense(neurons // 2, activation=activation))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [ ]:

pip install -U scikeras scikit-learn tensorflow


In [ ]:
# Setup and Run GridSearchCV
model = KerasClassifier(model=create_model, verbose=0)

param_grid = {
    "batch_size": [8, 16, 32],
    "epochs": [50, 100],
    "model__neurons": [32, 64],
    "model__activation": ["relu", "tanh"],
    "model__optimizer": ["adam", "rmsprop"]
}

grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, verbose=2)
grid_result = grid.fit(X_train, y_train)

print("Hyperparameter Tuning Completed.")


In [ ]:
# Show the best parameters 
print("\n the best parameters")
print(grid_result.best_params_)

print("Best Training accuracy")
print(grid_result.best_score_)

In [ ]:
# 4. Evaluation of Base Model vs Tuned Model

In [ ]:
# Evaluate Base Model
y_pred_base = (base_model.predict(X_test) > 0.5).astype(int)

print("\n Base Model Evaluation Metrics:")
print("Accuracy:", accuracy_score(y_test, y_pred_base))
print("Precision:", precision_score(y_test, y_pred_base))
print("Recall:", recall_score(y_test, y_pred_base))
print("F1 Score:", f1_score(y_test, y_pred_base))
print("\nClassification Report:\n", classification_report(y_test, y_pred_base))


In [ ]:
# Evaluate Best Tuned Model
best_model = grid_result.best_estimator_

y_pred_tuned = (best_model.predict(X_test) > 0.5).astype(int)

print("\n Tuned Model Evaluation Metrics:")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned))
print("Recall:", recall_score(y_test, y_pred_tuned))
print("F1 Score:", f1_score(y_test, y_pred_tuned))
print("\nClassification Report:\n", classification_report(y_test, y_pred_tuned))


In [ ]:
# 5. Summary and Comparison

In [ ]:
print("Base Model Accuracy:", accuracy_score(y_test, y_pred_base))
print("Tuned Model Accuracy:", accuracy_score(y_test, y_pred_tuned))


In [ ]:
print("- Hyperparameter tuning improved model performance by optimizing neurons, batch size, optimizer, and activation function.")
print("- Tuned model has better generalization and produces higher precision and recall, especially for detecting mines.")
print("- The structured Grid Search approach ensured a systematic evaluation of multiple ANN configurations.")
